# Fox scenes: make the lo-fi music
This makes the music for a lo-fi video with [ACE-Step 1.5](https://github.com/ACE-Step/ACE-Step-1.5), an open-source music model (MIT licence; its makers state the music may be used commercially). It runs on Google's free GPU and saves every track into your Google Drive.

1. **Runtime > Change runtime type > T4 GPU > Save.**
2. Set `SCENE`, then press the play button on the cell below and allow Google Drive access.
3. When it has finished, run `make-video.ipynb` with the same `SCENE` and `TYPE = "lofi"`.

When you upload a lo-fi video made with these tracks, answer **Yes** to YouTube's question about altered or synthetic (GenAI) content: its help page lists AI-generated music that is the main focus of a video as something to disclose, and says disclosing does not limit a video's audience or its eligibility to earn money.

In [ ]:
# Makes the lo-fi music for a fox scene with ACE-Step 1.5 and saves each track into your Google Drive as it is finished.
# Before pressing play: Runtime > Change runtime type > T4 GPU > Save.
# The first 15 minutes or so are set-up (about 15 GB of downloads); after that each three-minute track should take a minute or two.
# If the session stops part-way, press play again: tracks already in your Drive are kept and only the missing ones are made.

SCENE = "toymaker"               # "toymaker" or "stationmaster": the tempo and the instruments come from the scene's settings
TRACKS = 42                      # 42 tracks of three minutes fill a two-hour video with no track heard twice
DRIVE_FOLDER = "Cozy Scenes"     # the tracks go to My Drive / Cozy Scenes / music / <scene>


GEN = r'''
# Runs inside ACE-Step's own Python environment. Reads the list of tracks to make and writes each one as a FLAC file.
import json, os, shutil, sys, tempfile, time

JOB = json.load(open(sys.argv[1])); ROOT = JOB["ace"]
os.environ["ACESTEP_PROJECT_ROOT"] = ROOT
os.environ["ACESTEP_SAVE_MEMORY"] = "1"
os.environ["ACESTEP_GENERATION_TIMEOUT"] = "1800"
os.chdir(ROOT)

import torch
from acestep.handler import AceStepHandler
from acestep.inference import GenerationConfig, GenerationParams, generate_music

if not torch.cuda.is_available():
    sys.exit("No GPU. In Colab: Runtime > Change runtime type > T4 GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
dit = AceStepHandler()
status, ok = dit.initialize_service(project_root=ROOT, config_path="acestep-v15-turbo", device="cuda", use_flash_attention=False,
                                    compile_model=False, offload_to_cpu=False, offload_dit_to_cpu=False, quantization=None, prefer_source="huggingface")
print(status, flush=True)
if not ok:
    sys.exit("ACE-Step could not start (see the message above).")

made, tmp = 0, tempfile.mkdtemp()
for n, tr in enumerate(JOB["tracks"], 1):
    t0 = time.time()
    params = GenerationParams(task_type="text2music", caption=tr["caption"], lyrics="[Instrumental]", instrumental=True, vocal_language="unknown",
                              bpm=JOB["bpm"], keyscale=tr["key"], timesignature="4", duration=float(JOB["seconds"]), inference_steps=8, shift=3.0,
                              thinking=False, use_cot_metas=False, use_cot_caption=False, use_cot_language=False)
    config = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[tr["seed"]], audio_format="flac")
    result = generate_music(dit, None, params, config, save_dir=tmp)          # None: no language model, the music model alone
    if not result.success or not result.audios:
        print(f"[{n}/{len(JOB['tracks'])}] {tr['file']}: FAILED: {result.error} {result.status_message}", flush=True); continue
    audio = result.audios[0]; wav = audio["tensor"].detach().float().cpu(); sr = int(audio["sample_rate"])
    secs = wav.shape[-1] / sr; level = float(wav.pow(2).mean().sqrt())
    if not bool(torch.isfinite(wav).all()) or level < 0.005 or secs < 0.6 * JOB["seconds"]:
        print(f"[{n}/{len(JOB['tracks'])}] {tr['file']}: rejected (broken or silent: {secs:.0f} s, level {level:.4f})", flush=True); continue
    part = os.path.join(JOB["out"], tr["file"] + ".part")
    if audio.get("path") and os.path.exists(audio["path"]):
        shutil.copy(audio["path"], part); os.remove(audio["path"])
    else:
        import soundfile as sf
        sf.write(part, wav.transpose(0, 1).contiguous().numpy(), sr, format="FLAC", subtype="PCM_24")
    os.replace(part, os.path.join(JOB["out"], tr["file"])); made += 1
    print(f"[{n}/{len(JOB['tracks'])}] {tr['file']}: {secs:.0f} s of music in {time.time() - t0:.0f} s  (GPU memory peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB)", flush=True)
    del result, audio, wav; torch.cuda.empty_cache()
print(f"Made {made} of {len(JOB['tracks'])} tracks.")
sys.exit(0 if made == len(JOB["tracks"]) else 1)
'''

import json, os, subprocess, sys, urllib.request
from google.colab import drive
drive.mount("/content/drive")

BRANCH = "claude/inspiring-faraday-20xabu"
ACE, COMMIT = "/content/ACE-Step-1.5", "ca1e85fe9430179831e6bc6be790c332190a3866"     # the version of ACE-Step this was written against
T = json.load(urllib.request.urlopen(f"https://raw.githubusercontent.com/codingbythy/claude/refs/heads/{BRANCH}/fox-scenes/{SCENE}/scene.json"))["mix"]["lofi"]["tracks"]
dest = f"/content/drive/MyDrive/{DRIVE_FOLDER}/music/{SCENE}"
os.makedirs(dest, exist_ok=True)
tracks = [{"file": f"track-{i + 1:02d}.flac", "caption": T["captions"][i % len(T["captions"])], "key": T["keys"][(i * 3 + i // len(T["captions"])) % len(T["keys"])], "seed": T["seed"] + i} for i in range(TRACKS)]
todo = [t for t in tracks if not os.path.exists(f"{dest}/{t['file']}")]
print(f"{len(tracks) - len(todo)} of {TRACKS} tracks are already in My Drive / {DRIVE_FOLDER} / music / {SCENE}; {len(todo)} to make.")

if todo:
    if subprocess.run(["nvidia-smi"], capture_output=True).returncode:
        raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU > Save, then press play again.")
    env = {k: v for k, v in os.environ.items() if not k.startswith("UV_")}          # Colab's own settings for uv would send the install to the wrong place
    def sh(cmd, check=True):
        print("$", " ".join(cmd), flush=True)
        return subprocess.run(cmd, cwd=ACE, env=env, check=check).returncode
    if not os.path.isdir(ACE + "/.git"):
        os.makedirs(ACE, exist_ok=True)
        sh(["git", "init", "-q"]); sh(["git", "remote", "add", "origin", "https://github.com/ACE-Step/ACE-Step-1.5.git"])
        sh(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT]); sh(["git", "checkout", "-q", "FETCH_HEAD"])
    sh([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
    uv = [sys.executable, "-m", "uv"]
    frozen = ["--frozen"]
    if sh(uv + ["sync", "--frozen", "--no-dev", "--python", "3.12"], check=False):   # ACE-Step's own environment, beside Colab's, so nothing here needs restarting
        frozen = []; sh(uv + ["sync", "--no-dev", "--python", "3.12"])
    sh(uv + ["run"] + frozen + ["--no-dev", "acestep-download"], check=False)        # the model, about 10 GB (the next step fetches it too if this did not)
    with open("/content/gen_music.py", "w") as f:
        f.write(GEN)
    json.dump({"ace": ACE, "out": dest, "bpm": T["bpm"], "seconds": T["seconds"], "tracks": todo}, open("/content/music-job.json", "w"))
    code = sh(uv + ["run"] + frozen + ["--no-dev", "python", "-u", "/content/gen_music.py", "/content/music-job.json"], check=False)
    have = [t for t in tracks if os.path.exists(f"{dest}/{t['file']}")]
    json.dump({"made_with": "ACE-Step 1.5 (acestep-v15-turbo), MIT licence", "commit": COMMIT, "bpm": T["bpm"], "seconds": T["seconds"], "tracks": have}, open(f"{dest}/tracks.json", "w"), indent=1)
    print(f"{len(have)} of {TRACKS} tracks are now in My Drive / {DRIVE_FOLDER} / music / {SCENE}." + ("" if code == 0 else " Some could not be made: read the messages above, then press play again to retry only those."))
drive.flush_and_unmount()
print("Next: open make-video.ipynb, set the same SCENE with TYPE = \"lofi\", and it will use these tracks.")